In [ ]:
import pandas as pd
import re

# 1. Memuat data asli ulasan DANA
df = pd.read_csv('ulasan_tije.csv')
print('Dimensi data awal:', df.shape)

# 2. Fungsi khusus pembersihan
def clean_text(teks):
    if not isinstance(teks, str):
        return ""

    # a. Case Folding
    teks_clean = teks.lower()

    # b. Menghapus emoji/non-ASCII
    teks_clean = teks_clean.encode('ascii', 'ignore').decode('ascii')

    # c. Menghapus karakter spesial
    teks_clean = re.sub(r'[^a-z\s]', ' ', teks_clean)

    # d. Menghapus spasi ganda
    teks_clean = re.sub(r'\s+', ' ', teks_clean).strip()

    return teks_clean

# 3. Menerapkan cleaning
print("Sedang memproses pembersihan teks ulasan...")
df['content_clean'] = df['content'].apply(clean_text)

# e. Memfilter ulasan yang memiliki panjang minimal 4 karakter
df = df[df['content_clean'].str.len() >= 4].copy()

print(f"Dimensi setelah filter panjang teks: {df.shape}")
print("\nContoh hasil (5 baris teratas):")
print(df[['content', 'content_clean']].head())

Dimensi data awal: (4445, 1)
Sedang memproses pembersihan teks ulasan...
Dimensi setelah filter panjang teks: (4366, 2)

Contoh hasil (5 baris teratas):
                                             content  \
0  Aplikasi cukup membantu untuk cek rute dan pos...   
1  biasa naik jak 98. dan selalu muncul datanya d...   
2  Jujur ini ngebantu banget sih buat pejuang tra...   
3  Aplikasi Transjakarta ini jujur ngebantu bange...   
4  Transjakarta adalah alat yang sangat berguna (...   

                                       content_clean  
0  aplikasi cukup membantu untuk cek rute dan pos...  
1  biasa naik jak dan selalu muncul datanya di ap...  
2  jujur ini ngebantu banget sih buat pejuang tra...  
3  aplikasi transjakarta ini jujur ngebantu bange...  
4  transjakarta adalah alat yang sangat berguna d...  


In [ ]:
df.drop(columns=['content'], axis=1, inplace=True)
df.head()

,content_clean
0,aplikasi cukup membantu untuk cek rute dan pos...
1,biasa naik jak dan selalu muncul datanya di ap...
2,jujur ini ngebantu banget sih buat pejuang tra...
3,aplikasi transjakarta ini jujur ngebantu bange...
4,transjakarta adalah alat yang sangat berguna d...


In [ ]:
# Menyimpan hasil cleaning ke file CSV baru untuk kamu download
df.dropna()

output_clean_file = 'ulasan_tije_clean.csv'
df.to_csv(output_clean_file, index=False)
print("File berhasil diexport")

File berhasil diexport


In [ ]:
# 1. Memuat data ulasan TiJe
df = pd.read_csv('ulasan_tije_clean.csv')

# Mengisi baris kosong (jika ada ulasan yang terhapus total saat proses cleaning sebelumnya)
df['content_clean'] = df['content_clean'].fillna('')

print('Dimensi data:', df.shape)
print("\n5 Data Teratas Awal:")
print(df.head())

Dimensi data: (4366, 1)

5 Data Teratas Awal:
                                       content_clean
0  aplikasi cukup membantu untuk cek rute dan pos...
1  biasa naik jak dan selalu muncul datanya di ap...
2  jujur ini ngebantu banget sih buat pejuang tra...
3  aplikasi transjakarta ini jujur ngebantu bange...
4  transjakarta adalah alat yang sangat berguna d...


In [ ]:
# --- PENGEMBANGAN KAMUS LEXICON UNTUK KESEIMBANGAN DATA ---

positif = [
    # Dasar & Kepuasan
    'bagus', 'mantap', 'puas', 'cepat', 'rekomen', 'recommended', 'top', 'oke', 'ok', 'okk',
    'baik', 'memuaskan', 'nyaman', 'suka', 'love', 'best', 'hebat', 'keren', 'kerenn',
    'terbaik', 'lancar', 'aman', 'simpel', 'ringan', 'worth it', 'mudah', 'stabil', 'responsif',

    # Fungsi & Manfaat Aplikasi (Sering muncul di ulasan positif TiJe)
    'membantu', 'ngebantu', 'akurat', 'bantu', 'realtime', 'terbantu', 'bintang 5', 'bintang lima',
    'tertata', 'teratur', 'gampang', 'jelas', 'informatif', 'solutif', 'berfungsi', 'berguna',

    # Kata Serapan, Slang Ekspresif & Pujian Lainnya
    'naise', 'good', 'gege', 'wihh', 'joss', 'jos', 'wadidaw', 'awe', 'nice', 'cool',
    'makasih', 'terima kasih', 'terimakasih', 'thx', 'thanks', 'salut', 'hebring', 'jempol',
    'ajib', 'topcer', 'paten', 'maju terus', 'sukses', 'improving', 'happy', 'senang'
]

negatif = [
    # Dasar & Kekecewaan umum
    'buruk', 'jelek', 'lambat', 'lama', 'kecewa', 'mengecewakan', 'parah', 'parahh',
    'kapok', 'sampah', 'lemot', 'error', 'hilang', 'lag', 'loading', 'kurang',
    'sulit', 'gabisa', 'ga jelas', 'tidak sesuai', 'gak akurat', 'kurang akurat',

    # Isu Teknis & Finansial (Sektor krusial yang sering bikin ulasan jomplang ke negatif)
    'kedebet', 'debet', 'dongo', 'uninstall', 'bug', 'lelet', 'hadeh', 'komplen', 'complen',
    'terpotong', 'potong', 'saldo', 'hang', 'force close', 'crash', 'blank', 'gagal',
    'ribet', 'susah', 'pusing', 'muter', 'berputar', 'payah', 'kacau', 'down', 'nyendat',

    # Umpatan & Ekspresi Kekesalan Pengguna
    'bodoh', 'tolol', 'goblok', 'jeleknya', 'buruknya', 'nyesel', 'menyesal', 'benci',
    'parah banget', 'lemot banget', 'lama banget', 'rugi', 'kecewa banget', 'bobrok'
]

print('Jumlah kata kunci positif sekarang:', len(positif))
print('Jumlah kata kunci negatif sekarang:', len(negatif))

Jumlah kata kunci positif sekarang: 70
Jumlah kata kunci negatif sekarang: 62


In [ ]:
# 3. Fungsi Auto Labeling
def auto_label(teks):
    if not isinstance(teks, str):
        return 'netral'
    t = teks.lower()
    skor_pos = sum(1 for k in positif if k in t)
    skor_neg = sum(1 for k in negatif if k in t)

    if skor_pos > skor_neg:
        return 'positif'
    elif skor_neg > skor_pos:
        return 'negatif'
    else:
        return 'netral'

df['sentimen_auto'] = df['content_clean'].apply(auto_label)
print(df['sentimen_auto'].value_counts())
df.head(10)

sentimen_auto
positif    3222
netral      669
negatif     475
Name: count, dtype: int64


,content_clean,sentimen_auto
0,aplikasi cukup membantu untuk cek rute dan pos...,positif
1,biasa naik jak dan selalu muncul datanya di ap...,negatif
2,jujur ini ngebantu banget sih buat pejuang tra...,positif
3,aplikasi transjakarta ini jujur ngebantu bange...,positif
4,transjakarta adalah alat yang sangat berguna d...,positif
5,biasa naik jak dan jak dan selalu muncul datan...,negatif
6,bintang buat kemudahannya skema rutenya jelas ...,positif
7,membantu bgt utk tracking posisi bis jadi bisa...,positif
8,aplikasinya sangat membantu untuk cek rute dan...,positif
9,ini aplikasi bener ngebantu aku sebagai pendat...,positif


In [ ]:
# 4. Pisahkan baris untuk dikoreksi manual
sample = df.head(50).copy()

# Cara 1: simpan ke CSV, edit di Excel, lalu muat ulang
sample.to_csv('ulasan_tije_sample.csv', index=False)

# Setelah dikoreksi di Excel/Sheets, simpan kembali sebagai 'sudah_dikoreksi.csv' lalu muat:
# koreksi = pd.read_csv('sudah_dikoreksi.csv')

# Cara 2: koreksi inline di notebook (untuk demo)
# sample.loc[0, 'sentimen_auto'] = 'positif'   # contoh
# sample.loc[5, 'sentimen_auto'] = 'negatif'

In [ ]:
# 5. Menyimpan DataFrame yang telah diberi label otomatis ke file CSV baru
output_labeled_file = 'data_labeled.csv'
df.to_csv(output_labeled_file, index=False)

print(f"DataFrame dengan sentimen otomatis berhasil disimpan ke: '{output_labeled_file}'")

DataFrame dengan sentimen otomatis berhasil disimpan ke: 'data_labeled.csv'
